## CNN Model Testing (Izzie Lee)

In [ ]:
import torch
from torchvision import datasets, transforms

# load train and test dataset paths
train_path = '/Users/jiwonii/Desktop/Training'
test_path = '/Users/jiwonii/Desktop/Testing'

# transform for training images with data augmentation (resize, flips, rotation)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),  # 50% chance to flip images horizontally or vertically
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=20),  # randomly rotate by up to 20 degrees
    transforms.ToTensor()
])

# transform for testing images by converting to Tensors with 256x256 resize
test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# load train/test datasets and subfolder names and pass respective transforms
train_dataset = datasets.ImageFolder(root=train_path, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_path, transform=test_transform)

# image data loading in batches
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

print('Data Loaded Successfully...')

### Data Preview

Plots a batch of sample images from the training set to visually confirm the dataset loaded and labeled correctly.

In [ ]:
import matplotlib.pyplot as plt

# pulls a batch of images to display
images, labels = next(iter(train_loader))

# class names corresponding to the folder names
class_names = train_dataset.classes

# plot the first 10 images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(10):
    # PyTorch images (Channels, Height, Width) --> Matplotlib images (Height, Width, Channels)
    img = images[i].permute(1, 2, 0).numpy()

    axes[i].imshow(img)
    axes[i].set_title(class_names[labels[i]])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # cut image size by half

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)  # input size: 256x256
        self.bn1 = nn.BatchNorm2d(num_features=16)  # match out_channels

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)  # input size: 128x128
        self.bn2 = nn.BatchNorm2d(num_features=32)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)  # input size: 64x64
        self.bn3 = nn.BatchNorm2d(num_features=64)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)  # input size: 32x32
        self.bn4 = nn.BatchNorm2d(num_features=128)

        self.dropout = nn.Dropout()
        self.full_conn1 = nn.Linear(in_features=128 * 16 * 16, out_features=512)  # input size: 16x16
        self.full_conn2 = nn.Linear(in_features=512, out_features=4)  # output size: 4 classes

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # output size: 128x128
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # output size: 64x64
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # output size: 32x32
        x = self.pool(F.relu(self.bn4(self.conv4(x))))  # output size: 16x16

        x = torch.flatten(x, start_dim=1)  # flatten the tensor for fully connected layer
        x = F.relu(self.full_conn1(x))
        x = self.dropout(x)
        x = self.full_conn2(x)
        return x

model = SimpleCNN()
print(model)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cpu")
print(f"Using device: {device}")

# move the model to the chosen device (cpu)
model = model.to(device)

# define loss function (criterion), optimizer, and adaptive lr scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# set num of epochs (passes through the dataset)
epochs = 40

# init lists to track history of stats
train_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(epochs):
    # ------ TRAINING PHASE ------
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        # move data to the same device as model
        images, labels = images.to(device), labels.to(device)

        # reset/zero out the parameter gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # backward pass and optimize
        loss.backward()
        optimizer.step()

        # track training stats (error and accuracy)
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)  # _ represents the max logit value; we only care about predicted index value of logit
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    # track total training stats per epoch
    epoch_loss = running_loss / total_train
    epoch_acc = (correct_train / total_train) * 100

    # -------- TESTING PHASE --------
    model.eval()
    correct_test = 0
    total_test = 0

    with torch.no_grad():  # disable gradient calculation for efficiency, since no longer needed because not in training phase
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            # track testing stats (accuracy)
            _, predicted = torch.max(outputs, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    # track total testing accuracy per epoch
    test_acc = (correct_test / total_test) * 100

    # step scheduler based on test accuracy
    scheduler.step(test_acc)

    # save epoch stats to history lists
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    test_accuracies.append(test_acc)

    # print stats for each epoch, kept to 4 decimal places
    print(f"Epoch {epoch+1}/{epochs} "
          f"| Train Loss: {epoch_loss:.4f} "
          f"| Train Acc: {epoch_acc:.4f}% "
          f"| Test Acc: {test_acc:.4f}%")

print("Training finished!")

### Performance Visualization

This section plots the training loss and accuracy progress over all epochs. The graphs include:

* Tracking how well the model is minimizing its error.
* Comparing the training accuracy and testing accuracy.

In [ ]:
import matplotlib.pyplot as plt
epochs_range = range(1, epochs + 1)

# create a layout for plots
fig, (plot1, plot2) = plt.subplots(1, 2, figsize=(14, 5))

# plot 1: training loss
plot1.plot(epochs_range, train_losses, label='Training Loss', color='red', marker='o')
plot1.set_title('Training Loss over Epochs')
plot1.set_xlabel('Epochs')
plot1.set_ylabel('Loss')
plot1.set_xticks(epochs_range[::2])
plot1.grid(True, linestyle='--', alpha=0.6)
plot1.legend()

# plot 2: training and testing accuracy
plot2.plot(epochs_range, train_accuracies, label='Train Accuracy', color='blue', marker='o')
plot2.plot(epochs_range, test_accuracies, label='Test Accuracy', color='orange', marker='o')
plot2.set_title('Accuracy over Epochs')
plot2.set_xlabel('Epochs')
plot2.set_ylabel('Accuracy (%)')
plot2.set_xticks(epochs_range[::2])
plot2.grid(True, linestyle='--', alpha=0.6)
plot2.legend()

plt.tight_layout()
plt.show()